In [1]:
# -*- coding: utf-8 -*-
"""
Reviewer sensitivity tables for Hyb-Adam-UM.

This script reads already-produced Hyb-Adam-UM sensitivity CSV files and creates
compact LaTeX tables for the rebuttal/manuscript.

Main fixes in this version
--------------------------
1. The main initialization baseline is explicitly labeled:
       Baseline: row/column mean
   This corresponds to init_variant == "two_way_mean".

2. The "two_way_two_restarts" variant is excluded from the initialization table
   by default, because it changes the restart count and therefore is not purely
   an initialization sensitivity test.

3. Bolding is disabled by default. If BOLD_BEST=True, best values are bolded
   within each missingness block, not globally over the full table.

4. The initialization caption now states clearly that the baseline initialization
   is the row/column mean strategy used in the main experiments.

Expected input files
--------------------
Initialization sensitivity:
   hyb_adam_um_outputs/initialization_sensitivity/
       hyb_adam_um_initialization_sensitivity_detailed.csv

Hyperparameter sensitivity:
   hyb_adam_um_outputs/hyperparameter_sensitivity/
       hyb_adam_um_hyperparam_sensitivity_detailed.csv

Optional tree metric files:
   hyb_adam_um_outputs/initialization_sensitivity/
       hyb_adam_um_initialization_tree_metrics.csv

   hyb_adam_um_outputs/hyperparameter_sensitivity/
       hyb_adam_um_hyperparam_tree_metrics.csv

Expected merge keys for optional tree metrics
---------------------------------------------
Initialization:
   pct_missing, replicate, mask_seed, init_variant

Hyperparameter:
   pct_missing, replicate, mask_seed, sensitivity_variant

Outputs
-------
   hyb_adam_um_outputs/reviewer_sensitivity_tables/
       reviewer_initialization_sensitivity_table.tex
       reviewer_hyperparameter_sensitivity_table.tex
       reviewer_initialization_sensitivity_table_check.csv
       reviewer_hyperparameter_sensitivity_table_check.csv
       reviewer_sensitivity_interpretation.md
"""

from __future__ import annotations

import os
import re
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd


# ============================================================
# -------------------------- CONFIG ---------------------------
# ============================================================

OUTPUT_ROOT = Path("hyb_adam_um_outputs")

INIT_DIR = OUTPUT_ROOT / "initialization_sensitivity"
HYPER_DIR = OUTPUT_ROOT / "hyperparameter_sensitivity"
OUT_DIR = OUTPUT_ROOT / "reviewer_sensitivity_tables"

INIT_DETAILED_CSV = INIT_DIR / "hyb_adam_um_initialization_sensitivity_detailed.csv"
INIT_SUMMARY_CSV = INIT_DIR / "hyb_adam_um_initialization_sensitivity_summary.csv"

HYPER_DETAILED_CSV = HYPER_DIR / "hyb_adam_um_hyperparam_sensitivity_detailed.csv"
HYPER_SUMMARY_CSV = HYPER_DIR / "hyb_adam_um_hyperparam_sensitivity_summary.csv"

INIT_TREE_METRICS_CSV = INIT_DIR / "hyb_adam_um_initialization_tree_metrics.csv"
HYPER_TREE_METRICS_CSV = HYPER_DIR / "hyb_adam_um_hyperparam_tree_metrics.csv"

OUT_INIT_TEX = OUT_DIR / "reviewer_initialization_sensitivity_table.tex"
OUT_HYPER_TEX = OUT_DIR / "reviewer_hyperparameter_sensitivity_table.tex"
OUT_INIT_CHECK_CSV = OUT_DIR / "reviewer_initialization_sensitivity_table_check.csv"
OUT_HYPER_CHECK_CSV = OUT_DIR / "reviewer_hyperparameter_sensitivity_table_check.csv"
OUT_INTERPRETATION_MD = OUT_DIR / "reviewer_sensitivity_interpretation.md"

# In the main paper tables you use x 10^{-2}; keep this True for consistency.
SCALE_ERRORS_X100 = True

# For sensitivity tables, neutral presentation is usually better.
# If True, best values are bolded within each missingness group.
BOLD_BEST = False

# Keep initialization table pure: only initialization changes.
# If True, the row "Row/column mean + 2 restarts" will be included.
INCLUDE_RESTART_VARIANTS_IN_INIT_TABLE = False

# If False, the hyperparameter table excludes initialization/restart variants.
INCLUDE_INIT_VARIANTS_IN_HYPER_TABLE = False

# Include optional tree columns if found.
INCLUDE_TREE_METRICS_IF_AVAILABLE = True


# ============================================================
# ----------------------- LABELS / ORDERS ---------------------
# ============================================================

INIT_ORDER = [
    "observed_mean",
    "observed_median",
    "two_way_mean",
    "random_observed_range",
    "two_way_two_restarts",
]

INIT_DISPLAY = {
    "observed_mean": "Observed mean",
    "observed_median": "Observed median",
    "two_way_mean": "Baseline: row/column mean",
    "random_observed_range": "Random observed range",
    "two_way_two_restarts": "Row/column mean + 2 restarts",
}

INIT_RESTART_VARIANTS = {
    "two_way_two_restarts",
}

HYPER_ORDER = [
    "baseline",
    "lr_init_low",
    "lr_init_high",
    "schedule_fast_decay",
    "schedule_slow_decay",
    "clip_low",
    "clip_high",
    "clip_none",
    "fd_step_small",
    "fd_step_large",
]

HYPER_INIT_VARIANTS = [
    "old_global_mean_init",
    "two_restarts",
    "single_restart",
]

HYPER_DISPLAY = {
    "baseline": "Baseline",
    "lr_init_low": "Low initial LR",
    "lr_init_high": "High initial LR",
    "schedule_fast_decay": "Fast LR decay",
    "schedule_slow_decay": "Slow LR decay",
    "clip_low": "Clip = 1",
    "clip_high": "Clip = 10",
    "clip_none": "No clipping",
    "fd_step_small": r"FD step $5\cdot 10^{-6}$",
    "fd_step_large": r"FD step $10^{-4}$",
    "old_global_mean_init": "Global-mean init.",
    "two_restarts": "Two restarts",
    "single_restart": "Single restart",
}

METRIC_ALIASES: Dict[str, List[str]] = {
    "RMSE_miss": [
        "RMSE_miss", "rmse_miss", "RMSE_missing", "missing_RMSE",
    ],
    "MAE_miss": [
        "MAE_miss", "mae_miss", "MAE_missing", "missing_MAE",
    ],
    "Pearson_miss": [
        "Pearson_miss", "pearson_miss", "Pearson_missing",
    ],
    "Spearman_miss": [
        "Spearman_miss", "spearman_miss", "Spearman_missing",
    ],
    "Delta_final": [
        "Delta_final", "delta_final", "final_Delta", "Final_Delta",
        "Delta_total_completed", "delta_total_completed",
    ],
    "Delta_reduction_percent": [
        "Delta_reduction_percent", "delta_reduction_percent",
        "Delta_reduction_pct", "delta_reduction_pct",
    ],
    "runtime_seconds": [
        "runtime_seconds", "runtime_sec", "time_seconds", "Time_seconds",
    ],
    "RF_norm": [
        "RF_norm", "RF_normalized", "rf_norm", "rf_normalized",
        "RF_distance_norm", "RF_norm_vs_ML", "RF_normalized_vs_ML",
    ],
    "NJ_vs_ML_patristic_RMSE": [
        "NJ_vs_ML_patristic_RMSE", "NJ_vs_ML_patristic_rmse",
        "NJ_ML_patristic_RMSE", "NJ_ML_patristic_rmse",
        "patristic_RMSE", "patristic_rmse",
        "NJ_patristic_RMSE", "nj_ml_patristic_rmse",
    ],
}


# ============================================================
# -------------------------- HELPERS --------------------------
# ============================================================

def ensure_out_dir() -> None:
    OUT_DIR.mkdir(parents=True, exist_ok=True)


def normalize_name(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(name).lower())


def find_column(df: pd.DataFrame, aliases: Sequence[str]) -> Optional[str]:
    columns = list(df.columns)

    for alias in aliases:
        if alias in df.columns:
            return alias

    norm_to_col = {normalize_name(c): c for c in columns}
    for alias in aliases:
        key = normalize_name(alias)
        if key in norm_to_col:
            return norm_to_col[key]

    return None


def as_bool_series(s: pd.Series) -> pd.Series:
    if s.dtype == bool:
        return s.fillna(False)

    def one(x) -> bool:
        if pd.isna(x):
            return False
        if isinstance(x, (bool, np.bool_)):
            return bool(x)
        if isinstance(x, (int, float, np.integer, np.floating)):
            return bool(int(x))
        return str(x).strip().lower() in {
            "true", "1", "yes", "y", "success", "succeeded"
        }

    return s.map(one).astype(bool)


def finite_float(x) -> float:
    try:
        val = float(x)
    except Exception:
        return float("nan")
    return val if np.isfinite(val) else float("nan")


def mean_std(values: Iterable[object]) -> Tuple[float, float]:
    arr = pd.to_numeric(pd.Series(list(values)), errors="coerce").to_numpy(dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return float("nan"), float("nan")
    mean = float(np.mean(arr))
    std = float(np.std(arr, ddof=1)) if arr.size > 1 else 0.0
    return mean, std


def latex_escape_text(s: object) -> str:
    s = "" if s is None else str(s)

    # Preserve math expressions.
    if "$" in s:
        return s

    repl = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }
    return "".join(repl.get(ch, ch) for ch in s)


def fmt_pm(
    mean_value: object,
    std_value: object,
    decimals: int,
    scale: float = 1.0,
    bold: bool = False,
) -> str:
    mean = finite_float(mean_value)
    std = finite_float(std_value)

    if not np.isfinite(mean):
        return "--"

    mean *= scale

    if np.isfinite(std):
        std *= scale
        core = f"{mean:.{decimals}f} $\\pm$ {std:.{decimals}f}"
    else:
        core = f"{mean:.{decimals}f}"

    if bold:
        return r"\textbf{" + core + "}"

    return core


def fmt_success(row: pd.Series) -> str:
    n_success = int(finite_float(row.get("n_success", 0)) or 0)
    n_runs = int(finite_float(row.get("n_runs", n_success)) or n_success)
    return f"{n_success}/{n_runs}"


def is_all_nan(summary: pd.DataFrame, metric: str) -> bool:
    col = f"{metric}_mean"
    if col not in summary.columns:
        return True
    vals = pd.to_numeric(summary[col], errors="coerce")
    return not vals.notna().any()


def available_metric(summary: pd.DataFrame, metric: str) -> bool:
    return f"{metric}_mean" in summary.columns and not is_all_nan(summary, metric)


def lower_is_better(metric: str) -> bool:
    return metric not in {"Pearson_miss", "Spearman_miss"}


def metric_scale(metric: str) -> float:
    if not SCALE_ERRORS_X100:
        return 1.0
    if metric in {"RMSE_miss", "MAE_miss", "NJ_vs_ML_patristic_RMSE"}:
        return 100.0
    return 1.0


def metric_decimals(metric: str) -> int:
    if metric in {"RMSE_miss", "MAE_miss", "NJ_vs_ML_patristic_RMSE"} and SCALE_ERRORS_X100:
        return 2
    if metric in {"Pearson_miss", "Spearman_miss", "RF_norm"}:
        return 3
    if metric == "runtime_seconds":
        return 2
    if metric == "Delta_reduction_percent":
        return 1
    if metric == "Delta_final":
        return 2
    return 4


def metric_header(metric: str) -> str:
    if metric == "RMSE_miss":
        return r"RMSE$_{\rm miss}$" + (
            r" ($\times 10^{-2}$)" if SCALE_ERRORS_X100 else ""
        )
    if metric == "MAE_miss":
        return r"MAE$_{\rm miss}$" + (
            r" ($\times 10^{-2}$)" if SCALE_ERRORS_X100 else ""
        )
    if metric == "Pearson_miss":
        return r"Pearson$_{\rm miss}$"
    if metric == "Spearman_miss":
        return r"Spearman$_{\rm miss}$"
    if metric == "Delta_final":
        return r"Final $\Delta$"
    if metric == "Delta_reduction_percent":
        return r"$\Delta$ red. (\%)"
    if metric == "runtime_seconds":
        return r"Time (s)"
    if metric == "RF_norm":
        return r"RF$_{\rm norm}$"
    if metric == "NJ_vs_ML_patristic_RMSE":
        return r"NJ--ML patristic RMSE" + (
            r" ($\times 10^{-2}$)" if SCALE_ERRORS_X100 else ""
        )
    return latex_escape_text(metric)


def best_mask_within_group(
    df: pd.DataFrame,
    metric: str,
    idx,
    group_key: Optional[str] = None,
) -> bool:
    """
    Return True if row idx is best for metric.

    If group_key is given, best is computed only within that group.
    This fixes the previous global-bolding problem.
    """
    if not BOLD_BEST:
        return False

    col = f"{metric}_mean"
    if col not in df.columns:
        return False

    if group_key is not None and group_key in df.columns:
        group_value = df.loc[idx, group_key]
        sub = df[df[group_key] == group_value]
    else:
        sub = df

    vals = pd.to_numeric(sub[col], errors="coerce")
    finite = vals[np.isfinite(vals)]

    if finite.empty:
        return False

    best = finite.min() if lower_is_better(metric) else finite.max()
    current = finite_float(df.loc[idx, col])

    return bool(np.isfinite(current) and np.isclose(current, best, rtol=1e-10, atol=1e-12))


def merge_optional_metrics(
    df: pd.DataFrame,
    optional_csv: Path,
    keys: Sequence[str],
) -> Tuple[pd.DataFrame, str]:
    if not optional_csv.exists():
        return df, f"Optional tree-metric file not found: {optional_csv}"

    extra = pd.read_csv(optional_csv)

    missing_left = [k for k in keys if k not in df.columns]
    missing_right = [k for k in keys if k not in extra.columns]

    if missing_left or missing_right:
        return (
            df,
            "Could not merge optional tree metrics. "
            f"Missing left keys={missing_left}; missing right keys={missing_right}; "
            f"file={optional_csv}",
        )

    right_cols = list(keys)
    for _, aliases in METRIC_ALIASES.items():
        col = find_column(extra, aliases)
        if col is not None and col not in right_cols:
            right_cols.append(col)

    merged = df.merge(extra[right_cols], on=list(keys), how="left", suffixes=("", "_tree"))
    return merged, f"Merged optional tree metrics from: {optional_csv}"


# ============================================================
# ------------------------ SUMMARIZATION ----------------------
# ============================================================

def summarize_detailed(
    detailed: pd.DataFrame,
    group_cols: Sequence[str],
    optional_metric_csv: Optional[Path] = None,
    optional_metric_keys: Optional[Sequence[str]] = None,
) -> Tuple[pd.DataFrame, List[str]]:
    """
    Build mean/std summary from detailed per-run CSV.

    Means/stds are computed over successful runs only.
    n_runs/n_success/n_failed are computed over all rows.
    """
    warnings: List[str] = []
    df = detailed.copy()

    if optional_metric_csv is not None and optional_metric_keys is not None:
        df, msg = merge_optional_metrics(df, optional_metric_csv, optional_metric_keys)
        warnings.append(msg)

    for c in group_cols:
        if c not in df.columns:
            raise ValueError(f"Detailed CSV is missing required grouping column: {c}")

    if "success" in df.columns:
        success = as_bool_series(df["success"])
    else:
        success = pd.Series(True, index=df.index)
        warnings.append("No 'success' column found; treating all rows as successful.")

    df["__success__"] = success

    metric_cols: Dict[str, Optional[str]] = {}
    for metric, aliases in METRIC_ALIASES.items():
        metric_cols[metric] = find_column(df, aliases)

    rows: List[Dict[str, object]] = []

    for keys, group0 in df.groupby(list(group_cols), sort=False):
        if not isinstance(keys, tuple):
            keys = (keys,)

        success_mask = group0["__success__"].astype(bool)
        group = group0[success_mask].copy()

        row: Dict[str, object] = {c: v for c, v in zip(group_cols, keys)}
        row["n_runs"] = int(len(group0))
        row["n_success"] = int(success_mask.sum())
        row["n_failed"] = int(len(group0) - success_mask.sum())

        for meta in [
            "variant_description",
            "init_strategy",
            "n_restarts",
            "lr_init",
            "lr_milestones",
            "clip_grad_norm",
            "h_central_diff",
            "frac_missing_requested",
            "missingness_actual",
        ]:
            if meta in group0.columns:
                row[meta] = group0[meta].iloc[0]

        for metric, col in metric_cols.items():
            if col is None:
                row[f"{metric}_mean"] = float("nan")
                row[f"{metric}_std"] = float("nan")
                continue

            mean_val, std_val = mean_std(group[col])
            row[f"{metric}_mean"] = mean_val
            row[f"{metric}_std"] = std_val

        rows.append(row)

    return pd.DataFrame(rows), warnings


def normalize_summary_csv(
    summary: pd.DataFrame,
    required_cols: Sequence[str],
) -> Tuple[pd.DataFrame, List[str]]:
    warnings: List[str] = []
    df = summary.copy()

    for c in required_cols:
        if c not in df.columns:
            raise ValueError(f"Summary CSV is missing required column: {c}")

    for metric, aliases in METRIC_ALIASES.items():
        mean_col = find_column(df, [f"{a}_mean" for a in aliases] + [f"{metric}_mean"])
        std_col = find_column(df, [f"{a}_std" for a in aliases] + [f"{metric}_std"])

        if mean_col is not None:
            df[f"{metric}_mean"] = pd.to_numeric(df[mean_col], errors="coerce")
        else:
            df[f"{metric}_mean"] = np.nan

        if std_col is not None:
            df[f"{metric}_std"] = pd.to_numeric(df[std_col], errors="coerce")
        else:
            df[f"{metric}_std"] = np.nan

    if "n_runs" not in df.columns:
        if "n_success" in df.columns and "n_failed" in df.columns:
            df["n_runs"] = (
                pd.to_numeric(df["n_success"], errors="coerce").fillna(0).astype(int)
                + pd.to_numeric(df["n_failed"], errors="coerce").fillna(0).astype(int)
            )
        else:
            df["n_runs"] = np.nan

    if "n_success" not in df.columns:
        df["n_success"] = df["n_runs"]

    if "n_failed" not in df.columns:
        df["n_failed"] = 0

    return df, warnings


def load_initialization_summary() -> Tuple[pd.DataFrame, List[str]]:
    if INIT_DETAILED_CSV.exists():
        detailed = pd.read_csv(INIT_DETAILED_CSV)
        return summarize_detailed(
            detailed=detailed,
            group_cols=["pct_missing", "init_variant"],
            optional_metric_csv=INIT_TREE_METRICS_CSV,
            optional_metric_keys=["pct_missing", "replicate", "mask_seed", "init_variant"],
        )

    if INIT_SUMMARY_CSV.exists():
        summary = pd.read_csv(INIT_SUMMARY_CSV)
        return normalize_summary_csv(summary, required_cols=["pct_missing", "init_variant"])

    raise FileNotFoundError(
        "Initialization sensitivity results were not found. Tried:\n"
        f"  {INIT_DETAILED_CSV}\n"
        f"  {INIT_SUMMARY_CSV}\n\n"
        "Run the expensive Hyb-Adam-UM script first with "
        "RUN_INITIALIZATION_SENSITIVITY=True."
    )


def load_hyperparameter_summary() -> Tuple[pd.DataFrame, List[str]]:
    if HYPER_DETAILED_CSV.exists():
        detailed = pd.read_csv(HYPER_DETAILED_CSV)
        return summarize_detailed(
            detailed=detailed,
            group_cols=["sensitivity_variant"],
            optional_metric_csv=HYPER_TREE_METRICS_CSV,
            optional_metric_keys=[
                "pct_missing",
                "replicate",
                "mask_seed",
                "sensitivity_variant",
            ],
        )

    if HYPER_SUMMARY_CSV.exists():
        summary = pd.read_csv(HYPER_SUMMARY_CSV)
        return normalize_summary_csv(summary, required_cols=["sensitivity_variant"])

    raise FileNotFoundError(
        "Hyperparameter sensitivity results were not found. Tried:\n"
        f"  {HYPER_DETAILED_CSV}\n"
        f"  {HYPER_SUMMARY_CSV}\n\n"
        "Run the expensive Hyb-Adam-UM script first with "
        "RUN_HYPERPARAM_SENSITIVITY=True."
    )


# ============================================================
# ----------------------- TABLE PREP --------------------------
# ============================================================

def prepare_initialization_table(summary: pd.DataFrame) -> pd.DataFrame:
    df = summary.copy()

    if "pct_missing" not in df.columns:
        raise ValueError("Initialization summary must contain pct_missing.")

    if "init_variant" not in df.columns:
        raise ValueError("Initialization summary must contain init_variant.")

    df["pct_missing"] = pd.to_numeric(df["pct_missing"], errors="coerce")

    # Keep this table initialization-only.
    # The two-restart row changes the computational protocol, not only the initial guess.
    if not INCLUDE_RESTART_VARIANTS_IN_INIT_TABLE:
        df = df[~df["init_variant"].isin(INIT_RESTART_VARIANTS)].copy()

    df["Initialization"] = df["init_variant"].map(INIT_DISPLAY).fillna(df["init_variant"])
    df["__init_order__"] = (
        df["init_variant"].map({v: i for i, v in enumerate(INIT_ORDER)}).fillna(9999)
    )

    df = df.sort_values(["pct_missing", "__init_order__", "init_variant"]).reset_index(drop=True)
    return df.drop(columns=["__init_order__"], errors="ignore")


def prepare_hyperparameter_table(summary: pd.DataFrame) -> pd.DataFrame:
    df = summary.copy()

    if "sensitivity_variant" not in df.columns:
        raise ValueError("Hyperparameter summary must contain sensitivity_variant.")

    if not INCLUDE_INIT_VARIANTS_IN_HYPER_TABLE:
        df = df[~df["sensitivity_variant"].isin(HYPER_INIT_VARIANTS)].copy()

    order = HYPER_ORDER.copy()
    if INCLUDE_INIT_VARIANTS_IN_HYPER_TABLE:
        order = ["baseline"] + HYPER_INIT_VARIANTS + [
            v for v in HYPER_ORDER if v != "baseline"
        ]

    df["Variant"] = df["sensitivity_variant"].map(HYPER_DISPLAY).fillna(df["sensitivity_variant"])
    df["__order__"] = (
        df["sensitivity_variant"].map({v: i for i, v in enumerate(order)}).fillna(9999)
    )

    df = df.sort_values(["__order__", "sensitivity_variant"]).reset_index(drop=True)
    return df.drop(columns=["__order__"], errors="ignore")


def selected_init_metrics(df: pd.DataFrame) -> List[str]:
    metrics = ["RMSE_miss", "MAE_miss", "Delta_final"]

    if INCLUDE_TREE_METRICS_IF_AVAILABLE and available_metric(df, "RF_norm"):
        metrics.append("RF_norm")

    if INCLUDE_TREE_METRICS_IF_AVAILABLE and available_metric(df, "NJ_vs_ML_patristic_RMSE"):
        metrics.append("NJ_vs_ML_patristic_RMSE")

    metrics.append("runtime_seconds")
    return metrics


def selected_hyper_metrics(df: pd.DataFrame) -> List[str]:
    return ["RMSE_miss", "MAE_miss", "Delta_final", "runtime_seconds"]


# ============================================================
# ---------------------- LATEX WRITERS ------------------------
# ============================================================

def build_latex_table(
    df: pd.DataFrame,
    row_label_cols: Sequence[Tuple[str, str]],
    metrics: Sequence[str],
    caption: str,
    label: str,
    size_cmd: str = r"\scriptsize",
    bold_group_key: Optional[str] = None,
) -> str:
    colspec = "l" * len(row_label_cols) + "c" * (len(metrics) + 1)

    lines: List[str] = []
    lines.append(r"\begin{table}[t]")
    lines.append(r"\centering")
    lines.append(r"\caption{" + caption + "}")
    lines.append(r"\label{" + label + "}")
    lines.append(size_cmd)
    lines.append(r"\resizebox{\textwidth}{!}{%")
    lines.append(r"\begin{tabular}{" + colspec + "}")
    lines.append(r"\toprule")

    headers = [h for _, h in row_label_cols] + [metric_header(m) for m in metrics] + ["Success"]
    lines.append(" & ".join(headers) + r" \\")
    lines.append(r"\midrule")

    previous_group_value = None

    for idx, row in df.iterrows():
        if bold_group_key is not None and bold_group_key in df.columns:
            current_group_value = row.get(bold_group_key)
            if previous_group_value is not None and current_group_value != previous_group_value:
                # Optional visual separation between missingness blocks.
                # Comment this line if you prefer no inner midrules.
                # lines.append(r"\midrule")
                pass
            previous_group_value = current_group_value

        cells: List[str] = []

        for data_col, _ in row_label_cols:
            if data_col == "pct_missing":
                val = finite_float(row.get(data_col, np.nan))
                cells.append("--" if not np.isfinite(val) else f"{int(round(val))}\\%")
            else:
                cells.append(latex_escape_text(row.get(data_col, "")))

        for metric in metrics:
            cells.append(
                fmt_pm(
                    row.get(f"{metric}_mean", np.nan),
                    row.get(f"{metric}_std", np.nan),
                    decimals=metric_decimals(metric),
                    scale=metric_scale(metric),
                    bold=best_mask_within_group(
                        df=df,
                        metric=metric,
                        idx=idx,
                        group_key=bold_group_key,
                    ),
                )
            )

        cells.append(fmt_success(row))
        lines.append(" & ".join(cells) + r" \\")

    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")
    lines.append(r"}")
    lines.append(r"\end{table}")

    return "\n".join(lines) + "\n"


def build_initialization_latex(df: pd.DataFrame) -> str:
    metrics = selected_init_metrics(df)

    caption = (
        "Initialization sensitivity of Hyb-Adam-UM under fixed missing-entry masks. "
        "The baseline initialization is the row/column mean strategy used in the main "
        "experiments. Only the initialization of the missing entries is changed; the "
        "objective, optimizer settings, finite-difference step, clipping threshold, "
        "restart count, and number of epochs are kept fixed. Matrix errors are computed "
        "only on artificially hidden entries. Values are mean $\\pm$ standard deviation "
        "over fixed masks per missingness level."
    )

    if SCALE_ERRORS_X100:
        caption += " RMSE and MAE are reported as $\\times 10^{-2}$."

    return build_latex_table(
        df=df,
        row_label_cols=[
            ("pct_missing", "Missingness"),
            ("Initialization", "Initialization"),
        ],
        metrics=metrics,
        caption=caption,
        label="tab:hyb_initialization_sensitivity",
        size_cmd=r"\scriptsize",
        bold_group_key="pct_missing",
    )


def build_hyperparameter_latex(df: pd.DataFrame) -> str:
    metrics = selected_hyper_metrics(df)

    caption = (
        "Hyperparameter sensitivity of Hyb-Adam-UM at the fixed sensitivity missingness "
        "level. All variants are evaluated on the same frozen masks; only the indicated "
        "optimizer hyperparameter group is changed. Matrix errors are computed only on "
        "artificially hidden entries. Values are mean $\\pm$ standard deviation."
    )

    if SCALE_ERRORS_X100:
        caption += " RMSE and MAE are reported as $\\times 10^{-2}$."

    return build_latex_table(
        df=df,
        row_label_cols=[("Variant", "Variant")],
        metrics=metrics,
        caption=caption,
        label="tab:hyb_hyperparameter_sensitivity",
        size_cmd=r"\small",
        bold_group_key=None,
    )


# ============================================================
# --------------------- INTERPRETATION ------------------------
# ============================================================

def summarize_spread_by_missingness(init_df: pd.DataFrame) -> List[str]:
    lines: List[str] = []

    for pct, group in init_df.groupby("pct_missing", sort=True):
        success_group = group[
            pd.to_numeric(group["n_success"], errors="coerce").fillna(0) > 0
        ].copy()

        if success_group.empty:
            lines.append(f"- {int(pct)}% missing: no successful initialization runs.")
            continue

        rmse_vals = pd.to_numeric(success_group["RMSE_miss_mean"], errors="coerce")
        delta_vals = pd.to_numeric(success_group["Delta_final_mean"], errors="coerce")

        if rmse_vals.notna().any():
            best_rmse_row = success_group.loc[rmse_vals.idxmin()]
            worst_rmse_row = success_group.loc[rmse_vals.idxmax()]
            spread = float(
                worst_rmse_row["RMSE_miss_mean"] - best_rmse_row["RMSE_miss_mean"]
            )
            lines.append(
                f"- {int(pct)}% missing: best RMSE_miss = "
                f"{best_rmse_row['Initialization']} "
                f"({best_rmse_row['RMSE_miss_mean']:.6g}); "
                f"RMSE spread across initializations = {spread:.6g}."
            )

        if delta_vals.notna().any():
            best_delta_row = success_group.loc[delta_vals.idxmin()]
            lines.append(
                f"  Lowest final Delta: {best_delta_row['Initialization']} "
                f"({best_delta_row['Delta_final_mean']:.6g})."
            )

    return lines


def summarize_hyperparameter_spread(hyper_df: pd.DataFrame) -> List[str]:
    lines: List[str] = []

    success_df = hyper_df[
        pd.to_numeric(hyper_df["n_success"], errors="coerce").fillna(0) > 0
    ].copy()

    if success_df.empty:
        return ["- No successful hyperparameter-sensitivity runs."]

    rmse_vals = pd.to_numeric(success_df["RMSE_miss_mean"], errors="coerce")
    delta_vals = pd.to_numeric(success_df["Delta_final_mean"], errors="coerce")
    time_vals = pd.to_numeric(success_df["runtime_seconds_mean"], errors="coerce")

    if rmse_vals.notna().any():
        best = success_df.loc[rmse_vals.idxmin()]
        worst = success_df.loc[rmse_vals.idxmax()]
        lines.append(
            f"- Best mean RMSE_miss: {best['Variant']} "
            f"({best['RMSE_miss_mean']:.6g}); "
            f"worst: {worst['Variant']} ({worst['RMSE_miss_mean']:.6g})."
        )

    if delta_vals.notna().any():
        best = success_df.loc[delta_vals.idxmin()]
        lines.append(
            f"- Lowest mean final Delta: {best['Variant']} "
            f"({best['Delta_final_mean']:.6g})."
        )

    if time_vals.notna().any():
        fastest = success_df.loc[time_vals.idxmin()]
        lines.append(
            f"- Fastest mean runtime: {fastest['Variant']} "
            f"({fastest['runtime_seconds_mean']:.6g} s)."
        )

    baseline = success_df[success_df["sensitivity_variant"] == "baseline"]
    if len(baseline) == 1 and rmse_vals.notna().any():
        base = float(baseline.iloc[0]["RMSE_miss_mean"])
        if np.isfinite(base) and abs(base) > 1e-15:
            rel = []
            for _, row in success_df.iterrows():
                if row["sensitivity_variant"] == "baseline":
                    continue
                val = finite_float(row["RMSE_miss_mean"])
                if np.isfinite(val):
                    rel.append(abs(100.0 * (val - base) / base))
            if rel:
                lines.append(
                    "- Maximum absolute RMSE_miss change relative to baseline: "
                    f"{max(rel):.2f}%."
                )

    return lines


def write_interpretation(
    init_df: pd.DataFrame,
    hyper_df: pd.DataFrame,
    warnings: Sequence[str],
) -> None:
    lines: List[str] = []
    lines.append("# Reviewer sensitivity tables\n\n")

    lines.append("## Files generated\n\n")
    lines.append(f"- Initialization table: `{OUT_INIT_TEX}`\n")
    lines.append(f"- Hyperparameter table: `{OUT_HYPER_TEX}`\n")
    lines.append(f"- Initialization check CSV: `{OUT_INIT_CHECK_CSV}`\n")
    lines.append(f"- Hyperparameter check CSV: `{OUT_HYPER_CHECK_CSV}`\n\n")

    lines.append("## Initialization sensitivity: data-driven summary\n\n")
    lines.extend(x + "\n" for x in summarize_spread_by_missingness(init_df))
    lines.append("\n")

    lines.append("## Hyperparameter sensitivity: data-driven summary\n\n")
    lines.extend(x + "\n" for x in summarize_hyperparameter_spread(hyper_df))
    lines.append("\n")

    missing_tree = []
    if not available_metric(init_df, "RF_norm"):
        missing_tree.append("RF_norm")
    if not available_metric(init_df, "NJ_vs_ML_patristic_RMSE"):
        missing_tree.append("NJ_vs_ML_patristic_RMSE")

    if missing_tree:
        lines.append("## Important note about tree metrics\n\n")
        lines.append(
            "The initialization table could not include the following tree metrics because "
            "they were not present in the sensitivity CSV files: "
            + ", ".join(missing_tree)
            + ". The Hyb-Adam-only sensitivity runner computes matrix/objective metrics, "
            "not RF/patristic tree metrics. To include tree sensitivity metrics, save "
            "completed sensitivity matrices and run the tree-comparison pipeline on those "
            "matrices, then merge using the documented keys.\n\n"
        )

    if warnings:
        lines.append("## Warnings / merge notes\n\n")
        for w in warnings:
            lines.append(f"- {w}\n")

    OUT_INTERPRETATION_MD.write_text("".join(lines), encoding="utf-8")


# ============================================================
# ---------------------------- MAIN ---------------------------
# ============================================================

def main() -> None:
    ensure_out_dir()

    init_summary, init_warnings = load_initialization_summary()
    hyper_summary, hyper_warnings = load_hyperparameter_summary()

    init_table_df = prepare_initialization_table(init_summary)
    hyper_table_df = prepare_hyperparameter_table(hyper_summary)

    init_table_df.to_csv(OUT_INIT_CHECK_CSV, index=False)
    hyper_table_df.to_csv(OUT_HYPER_CHECK_CSV, index=False)

    OUT_INIT_TEX.write_text(build_initialization_latex(init_table_df), encoding="utf-8")
    OUT_HYPER_TEX.write_text(build_hyperparameter_latex(hyper_table_df), encoding="utf-8")

    write_interpretation(
        init_df=init_table_df,
        hyper_df=hyper_table_df,
        warnings=[*init_warnings, *hyper_warnings],
    )

    print("Saved:")
    print(f"  {OUT_INIT_TEX}")
    print(f"  {OUT_HYPER_TEX}")
    print(f"  {OUT_INIT_CHECK_CSV}")
    print(f"  {OUT_HYPER_CHECK_CSV}")
    print(f"  {OUT_INTERPRETATION_MD}")

    if not INCLUDE_RESTART_VARIANTS_IN_INIT_TABLE:
        print()
        print("Note: restart-changing initialization variants were excluded from the")
        print("initialization table to keep the caption mathematically correct.")

    if not available_metric(init_table_df, "RF_norm") or not available_metric(
        init_table_df, "NJ_vs_ML_patristic_RMSE"
    ):
        print()
        print("Warning: RF/patristic tree metrics were not found in the sensitivity CSV files.")
        print("The LaTeX table was still generated, but it contains only matrix/objective metrics.")
        print("See reviewer_sensitivity_interpretation.md for the exact note.")


if __name__ == "__main__":
    main()

Saved:
  hyb_adam_um_outputs/reviewer_sensitivity_tables/reviewer_initialization_sensitivity_table.tex
  hyb_adam_um_outputs/reviewer_sensitivity_tables/reviewer_hyperparameter_sensitivity_table.tex
  hyb_adam_um_outputs/reviewer_sensitivity_tables/reviewer_initialization_sensitivity_table_check.csv
  hyb_adam_um_outputs/reviewer_sensitivity_tables/reviewer_hyperparameter_sensitivity_table_check.csv
  hyb_adam_um_outputs/reviewer_sensitivity_tables/reviewer_sensitivity_interpretation.md

Note: restart-changing initialization variants were excluded from the
initialization table to keep the caption mathematically correct.

The LaTeX table was still generated, but it contains only matrix/objective metrics.
See reviewer_sensitivity_interpretation.md for the exact note.
